# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders (4).csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [2]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [16]:
# 1
print('Количество строк и столбцов: ',df.shape)
print('Типы столбцов в дата фрейме: ',df.dtypes) 
print('Пропусков в "delivery_days" :', df['delivery_days'].isna().sum())
print('Количество уникальных значений в столбце "city"' ,df["city"].nunique())
print('Количество уникальных значений в столбце "category"' ,df["category"].nunique())
print('Количество уникальных значений в столбце "status"' ,df["status"].nunique())
print('Количество уникальных значений в столбце "customer_id"' ,df["customer_id"].nunique())
print('Название городов в датафрейме: ',df['city'].unique())
print(df['status'].value_counts())

Количество строк и столбцов:  (5463, 8)
Типы столбцов в дата фрейме:  order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object
Пропусков в "delivery_days" : 160
Количество уникальных значений в столбце "city" 14
Количество уникальных значений в столбце "category" 5
Количество уникальных значений в столбце "status" 2
Количество уникальных значений в столбце "customer_id" 2003
Название городов в датафрейме:  ['Алматы' 'Астана' 'Караганда' 'Шымкент' 'алматы' 'Атырау' 'шымкент'
 'Shymkent' 'астана' 'Актобе' 'Astana ' 'АЛМАТЫ' ' Алматы' 'Almaty']
status
оплачен    4904
отменён     559
Name: count, dtype: int64


#### Три проблемы данных для менеджера

1. **Пропущенные значения:** в столбце `delivery_days` имеется ровно 160 пропусков.Данная проблема может привести к затруднениям анализу доставок и службы доставки.

2. **Разные написания названий городов:** в столбце `city`  один город записан несколькими способами, например `Алматы`, `алматы`, `АлМаты` и `Almaty`. Данная проблема может привести к ошибкам при дальнейшем анализе.

3. **Отменённые заказы:** в столбце `status` имеется 559 отменённых заказов и 4904 оплаченных. При расчете общей выручки нужно учитывать этот факт.

## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [22]:
orders = df.loc[(df['status']=='оплачен')& (df['customer_id']!='TEST'), "amount"]

print(orders)
print("Среднее заказов :", orders.mean())
print("Медиана заказов :", orders.median())
print("25 percentile of the orders is :", orders.quantile(0.25))
print("75 percentile of the orders is :", orders.quantile(0.75))

0        8070.0
1        3630.0
2        9120.0
3        9350.0
4       11010.0
         ...   
5457     9170.0
5458    11430.0
5459     3640.0
5460    22150.0
5462    14910.0
Name: amount, Length: 4901, dtype: float64
Среднее заказов : 14672.391348704346
Медиана заказов : 10300.0
25 percentile of the orders is : 5880.0
75 percentile of the orders is : 18690.0


**Вывод:**

Медиана заказов равна 10 300, в то время как среднее значение — 14 672,39. Среднее сущесственно выше медианы , это означает что имеются дорогие заказы которые 'тянут' среднее значение вверх.

Честнее показать в отчете медиану, так-как медиана является менее чуствительной к очень большим значениям

Также 50% заказов находятся в диапазоне от 5 880 до 18 690 (между 25-м и 75-м перцентилями).

В отчёте можно использовать оба показателя , но медиану использовать как основной показатель типичного заказа будет лучше.

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [32]:
cncld_ordrs = df["status"] == "отменён"
a=cncld_ordrs.mean()
shmknt=df['city'].isin(["Шымкент", "шымкент", "Shymkent"]) # creates a boolean mask for the Shymkent city.
b=cncld_ordrs[shmknt].mean()

almty = df["city"].isin(["Алматы", "алматы", "АлМаты", "АЛМАТЫ", "Almaty"])
c = cncld_ordrs[almty].mean()
print("Все заказы:", a)
print("Шымкент:", b)
print("Алматы:", c)

d = b / c
print('Отмен в Шымкенте больше чем в Алматы в  ', round(d,3), "раза")

Все заказы: 0.1023247300018305
Шымкент: 0.20398009950248755
Алматы: 0.07964184731385486
Отмен в Шымкенте больше чем в Алматы в   2.561 раза


Процент отмененных заказов в Шымкенте равен 20,39% , в то время как в Алматы - 7,96% . в Шымкенте доля отмен примерно в 2,56 раза выше, чем в Алматы. Общий процент отмен составляет 10,23%.



## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [ ]:
# 4

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [ ]:
# 5

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [ ]:
# 6

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8